## Movie Recommender System 

**Dataset:** TMDB 5000 Movies + TMDB 5000 Credits (`raw_data/`)
**Technique:** Content-Based Filtering
`movie details -> tags -> stemming -> Bag-of-Words vectors -> Cosine Similarity`

**Libraries used**
- `numpy`, `pandas` : data handling
- `ast` : convert JSON-like text into python lists
- `nltk` (PorterStemmer) : reduce words to root form
- `scikit-learn` : `CountVectorizer` (text -> vectors), `cosine_similarity`
- `pickle` : save files for the Streamlit app

**Output files:** `movie_dict.pkl`, `similarity.pkl` (+ `movies.pkl`) -> used by `recommender.py` and `frontend/app.py`

**Flow of this notebook**
1. Import  2. Load data  3. Merge & select columns  4. Data cleaning  5. Extract features
6. Create tags  7. Stemming  8. Text to vectors  9. Cosine similarity  10. Recommend function  11. Save files

## 1. Import Basic Libraries

In [1]:
# numpy: arrays | pandas: tables/dataframes
import numpy as np
import pandas as pd

## 2. Load Dataset

In [2]:
# Check: the csv path is a FOLDER that contains the csv file (so the path is written twice below)
import os
p = '../raw_data/tmdb_5000_movies.csv'
print(os.path.isdir(p))   # True = it's a folder
print(os.listdir(p) if os.path.isdir(p) else "it's a file")


True
['tmdb_5000_movies.csv']


In [3]:
# Load both csv files (folder name + file name)
movies = pd.read_csv('../raw_data/tmdb_5000_movies.csv/tmdb_5000_movies.csv')
credits = pd.read_csv('../raw_data/tmdb_5000_credits.csv/tmdb_5000_credits.csv')

In [4]:
# (rows, columns) of movies file
movies.shape

(4803, 20)

In [5]:
# (rows, columns) of credits file
credits.shape

(4803, 4)

In [6]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [7]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


## 3. Merge Both Files & Select Important Columns

In [8]:
# Join both tables using the common column 'title'
movies = movies.merge(credits, on='title')

In [9]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [10]:
# IMP features:
# genres
# movies_id
# keywords
# title
# overview
# cast
# crew

In [11]:
# Keep only the 7 useful columns
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [12]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## 4. Data Cleaning (missing values & duplicates)

In [13]:
# Count missing values in each column
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [14]:
# Remove rows having missing values (only 'overview' had 3)
movies.dropna(inplace=True)

In [15]:
movies.isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [16]:
# Check for duplicate rows
movies.duplicated().sum()

np.int64(0)

In [17]:
# (ADDED) Dropped rows leave gaps in row numbers; reset so row number = similarity row
movies = movies.reset_index(drop=True)

## 5. Feature Extraction (JSON text -> simple lists)

In [18]:
# We are making 3 main colums
# movie_id,title,tags(overview + genres + keywords + cast +	crew)
# preparing genres + keywords + cast +	crew for adding in overview

In [19]:
# ast converts text like '[{...}]' into a real python list
import ast

In [20]:
# Helper: genres/keywords JSON text -> list of names only
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [21]:
# Apply on genres
movies['genres'] = movies['genres'].apply(convert)

In [22]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [23]:
# Apply on keywords
movies['keywords'] = movies['keywords'].apply(convert)

In [24]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [25]:
# Helper: take only the first 3 cast members (main actors)
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

In [26]:
# Apply on cast
movies['cast'] = movies['cast'].apply(convert3)

In [27]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [28]:
# Helper: find the Director's name inside crew list
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [29]:
# Apply on crew
movies['crew'] = movies['crew'].apply(fetch_director)

In [30]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [31]:
# Split overview sentence into list of words (so it can be added with other lists)
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [32]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [33]:
# Remove spaces inside names: 'Sam Worthington' -> 'SamWorthington' (stays ONE tag)
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [34]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]


## 6. Create Tags (overview + genres + keywords + cast + crew)

In [35]:
# Join all columns into one list of words called 'tags'
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [36]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [37]:
# Final dataframe with 3 columns only
new_df = movies[['movie_id','title','tags']]

In [38]:
new_df

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...
4801,9367,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui..."
4802,72766,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended..."
4803,231617,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,..."
4804,126186,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is..."


In [39]:
# List of words -> one single text string
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [40]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [41]:
new_df['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [42]:
# Convert everything to lowercase
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [43]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


## 7. Stemming (reduce words to root form)

In [44]:
# nltk library for stemming
import nltk

In [45]:
# PorterStemmer reduces words to root: loved, loving -> love
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [46]:
# Helper: stem every word of a text
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [47]:
# Apply stemming on tags
new_df['tags'] = new_df['tags'].apply(stem)

## 8. Convert Text to Vectors (Bag of Words)

In [48]:
# CountVectorizer: count the 5000 most frequent words, ignore stop words (the, is, a ...)
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [49]:
# Each movie becomes a vector of 5000 numbers
vectors = cv.fit_transform(new_df['tags']).toarray()

In [50]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4806, 5000))

In [51]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [52]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

## 9. Cosine Similarity (distance between movies)

In [53]:
# calculate cosine similaritydistance(angle between vectors)

In [54]:
# Cosine similarity compares two movie vectors (1 = very similar)
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
# Compare every movie with every other movie (4806 x 4806 table)
similarity = cosine_similarity(vectors)

In [56]:
print(similarity.shape)

(4806, 4806)


## 10. Recommendation Function (top 5 similar movies)

In [57]:
# Takes a movie name, finds its row, sorts similarity scores high -> low, prints top 5 (skipping itself)
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [58]:
# Test the function
recommend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


## 11. Save Files for the App

In [59]:
# pickle saves python objects into files
import pickle

In [60]:
# (ADDED) Optional: float32 makes similarity.pkl ~92 MB instead of ~185 MB (GitHub limit is 100 MB)
similarity = similarity.astype('float32')

In [61]:
# Save dataframe
pickle.dump(new_df,open('movies.pkl','wb'))

In [62]:
# Save as dictionary (this file is used by recommender.py / app)
pickle.dump(new_df.to_dict(), open('movie_dict.pkl','wb'))

In [63]:
# Save similarity matrix (this file is used by recommender.py / app)
pickle.dump(similarity,open('similarity.pkl','wb'))